# Industrial Machine Condition Classification

## 1. Install libraries

In [ ]:
!pip -q install catboost lightgbm xgboost

## 2. Load and inspect the CSV files

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

TRAIN_PATH = "/content/student_dataset.csv"
TEST_PATH = "/content/test.csv"
SAMPLE_PATH = "/content/sample_submission.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_PATH)

print("Training shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Sample submission shape:", sample_submission.shape)

display(train_df.head())
display(test_df.head())
display(sample_submission.head())

print("\nTraining missing values:")
print(train_df.isnull().sum())

print("\nTarget distribution:")
print(train_df["condition"].value_counts().sort_index())

## 3. Prepare features and create a validation split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from catboost import CatBoostClassifier

# unit_id is an identifier, so exclude it from the initial model.
X = train_df.drop(columns=["condition", "unit_id"]).copy()
y = train_df["condition"].copy()

# CatBoost handles categorical values directly; represent missing types explicitly.
X["unit_type"] = X["unit_type"].fillna("Unknown").astype(str)
cat_features = ["unit_type"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training rows:", len(X_train))
print("Validation rows:", len(X_valid))

## 4. Baseline CatBoost model

In [ ]:
baseline_model = CatBoostClassifier(
    iterations=1500,
    depth=6,
    learning_rate=0.05,
    loss_function="MultiClass",
    eval_metric="Accuracy",
    cat_features=cat_features,
    random_seed=42,
    early_stopping_rounds=100,
    verbose=100,
    allow_writing_files=False
)

baseline_model.fit(
    X_train, y_train,
    eval_set=(X_valid, y_valid),
    use_best_model=True
)

baseline_pred = baseline_model.predict(X_valid).ravel().astype(int)
baseline_accuracy = accuracy_score(y_valid, baseline_pred)
print(f"Baseline CatBoost validation accuracy: {baseline_accuracy:.5f}")

print(classification_report(
    y_valid, baseline_pred,
    labels=[0, 1, 2],
    target_names=["Normal", "Thermal Failure", "Mechanical Failure"],
    zero_division=0
))

ConfusionMatrixDisplay.from_predictions(
    y_valid, baseline_pred,
    labels=[0, 1, 2],
    display_labels=["Normal", "Thermal Failure", "Mechanical Failure"],
    values_format="d", cmap="Blues"
)
plt.tight_layout()
plt.show()

## 5. Tune baseline CatBoost hyperparameters

In [ ]:
# These experiments use the original features (before feature engineering).
configs = [
    {"depth": 4, "learning_rate": 0.05, "l2_leaf_reg": 3},
    {"depth": 5, "learning_rate": 0.05, "l2_leaf_reg": 5},
    {"depth": 6, "learning_rate": 0.03, "l2_leaf_reg": 5},
    {"depth": 7, "learning_rate": 0.03, "l2_leaf_reg": 5},
    {"depth": 8, "learning_rate": 0.03, "l2_leaf_reg": 7},
]

baseline_tuning_results = []
for params in configs:
    candidate = CatBoostClassifier(
        iterations=2000,
        loss_function="MultiClass",
        eval_metric="Accuracy",
        cat_features=cat_features,
        random_seed=42,
        early_stopping_rounds=120,
        verbose=False,
        allow_writing_files=False,
        **params
    )
    candidate.fit(X_train, y_train, eval_set=(X_valid, y_valid), verbose=False)
    pred = candidate.predict(X_valid).ravel().astype(int)
    baseline_tuning_results.append({
        **params,
        "validation_accuracy": accuracy_score(y_valid, pred),
        "best_iteration": candidate.get_best_iteration()
    })

display(pd.DataFrame(baseline_tuning_results).sort_values(
    "validation_accuracy", ascending=False
).reset_index(drop=True))

## 6. Feature engineering and engineered CatBoost

In [ ]:
def add_features(df):
    """Add domain-inspired features derived from existing sensor measurements."""
    df = df.copy()
    df["temperature_difference"] = df["process_temp"] - df["ambient_temp"]
    df["torque_speed_product"] = df["drive_torque"] * df["shaft_speed"] / 1000
    df["coolant_per_temperature_difference"] = (
        df["coolant_flow"] / (df["temperature_difference"].abs() + 1)
    )
    df["wear_vibration_interaction"] = df["component_wear"] * df["vibration_index"]
    return df

X_train_eng = add_features(X_train)
X_valid_eng = add_features(X_valid)

engineered_model = CatBoostClassifier(
    iterations=2000,
    depth=6,
    learning_rate=0.03,
    l2_leaf_reg=5,
    loss_function="MultiClass",
    eval_metric="Accuracy",
    cat_features=cat_features,
    random_seed=42,
    early_stopping_rounds=120,
    verbose=False,
    allow_writing_files=False
)

engineered_model.fit(
    X_train_eng, y_train,
    eval_set=(X_valid_eng, y_valid),
    verbose=False
)

eng_pred = engineered_model.predict(X_valid_eng).ravel().astype(int)
eng_prob = engineered_model.predict_proba(X_valid_eng)
eng_accuracy = accuracy_score(y_valid, eng_pred)

print(f"Baseline CatBoost accuracy:   {baseline_accuracy:.5f}")
print(f"Engineered CatBoost accuracy: {eng_accuracy:.5f}")
print(f"Difference:                   {eng_accuracy - baseline_accuracy:+.5f}")

importance = pd.DataFrame({
    "Feature": X_train_eng.columns,
    "Importance": engineered_model.feature_importances_
}).sort_values("Importance", ascending=False)
display(importance)

plt.figure(figsize=(9, 6))
plt.barh(importance["Feature"][::-1], importance["Importance"][::-1])
plt.xlabel("Feature importance")
plt.title("Engineered CatBoost feature importance")
plt.tight_layout()
plt.show()

## 7. Additional CatBoost hyperparameter experiments

In [ ]:
# Further tuning, now using the engineered feature set.
advanced_configs = [
    {"depth": 4, "learning_rate": 0.05, "l2_leaf_reg": 3, "random_strength": 1, "bagging_temperature": 1},
    {"depth": 5, "learning_rate": 0.03, "l2_leaf_reg": 3, "random_strength": 1, "bagging_temperature": 1},
    {"depth": 6, "learning_rate": 0.03, "l2_leaf_reg": 3, "random_strength": 1, "bagging_temperature": 1},
    {"depth": 6, "learning_rate": 0.03, "l2_leaf_reg": 7, "random_strength": 1, "bagging_temperature": 1},
    {"depth": 7, "learning_rate": 0.03, "l2_leaf_reg": 5, "random_strength": 1, "bagging_temperature": 1},
    {"depth": 8, "learning_rate": 0.03, "l2_leaf_reg": 5, "random_strength": 1, "bagging_temperature": 1},
    {"depth": 6, "learning_rate": 0.03, "l2_leaf_reg": 5, "random_strength": 0.5, "bagging_temperature": 0.5},
    {"depth": 6, "learning_rate": 0.03, "l2_leaf_reg": 5, "random_strength": 2, "bagging_temperature": 2},
    {"depth": 7, "learning_rate": 0.02, "l2_leaf_reg": 7, "random_strength": 1, "bagging_temperature": 0.5},
    {"depth": 5, "learning_rate": 0.05, "l2_leaf_reg": 5, "random_strength": 0.5, "bagging_temperature": 2},
]

advanced_results = []
for i, params in enumerate(advanced_configs, start=1):
    candidate = CatBoostClassifier(
        iterations=2500,
        loss_function="MultiClass",
        eval_metric="Accuracy",
        cat_features=cat_features,
        random_seed=42,
        early_stopping_rounds=150,
        verbose=False,
        allow_writing_files=False,
        **params
    )
    candidate.fit(X_train_eng, y_train, eval_set=(X_valid_eng, y_valid), verbose=False)
    pred = candidate.predict(X_valid_eng).ravel().astype(int)
    advanced_results.append({
        **params,
        "validation_accuracy": accuracy_score(y_valid, pred),
        "best_iteration": candidate.get_best_iteration()
    })

display(pd.DataFrame(advanced_results).sort_values(
    "validation_accuracy", ascending=False
).reset_index(drop=True))
print("Best engineered CatBoost score from the main experiment:", round(eng_accuracy, 5))

## 8. Alternative model experiment: LightGBM

In [ ]:
from lightgbm import LGBMClassifier

X_train_lgb = X_train.copy()
X_valid_lgb = X_valid.copy()
X_train_lgb["unit_type"] = X_train_lgb["unit_type"].astype("category")
X_valid_lgb["unit_type"] = pd.Categorical(
    X_valid_lgb["unit_type"], categories=X_train_lgb["unit_type"].cat.categories
)

lgb_model = LGBMClassifier(
    objective="multiclass",
    n_estimators=1200,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    reg_lambda=2,
    colsample_bytree=0.9,
    subsample=0.8,
    subsample_freq=1,
    random_state=42,
    verbosity=-1
)
lgb_model.fit(X_train_lgb, y_train, categorical_feature=["unit_type"])
lgb_pred = lgb_model.predict(X_valid_lgb)
lgb_accuracy = accuracy_score(y_valid, lgb_pred)
print(f"LightGBM validation accuracy (original features): {lgb_accuracy:.5f}")
print(f"Engineered CatBoost validation accuracy:          {eng_accuracy:.5f}")

## 9. Alternative model experiment: XGBoost

In [ ]:
from xgboost import XGBClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_cols = ["unit_type"]
numerical_cols = [c for c in X_train_eng.columns if c not in categorical_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("numerical", "passthrough", numerical_cols)
    ]
)
X_train_transformed = preprocessor.fit_transform(X_train_eng)
X_valid_transformed = preprocessor.transform(X_valid_eng)

xgb_classifier = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    n_estimators=1000,
    max_depth=6,
    learning_rate=0.03,
    min_child_weight=3,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=3,
    reg_alpha=0.1,
    eval_metric="mlogloss",
    early_stopping_rounds=100,
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)
xgb_classifier.fit(
    X_train_transformed, y_train,
    eval_set=[(X_valid_transformed, y_valid)],
    verbose=False
)
xgb_pred = xgb_classifier.predict(X_valid_transformed).astype(int)
xgb_prob = xgb_classifier.predict_proba(X_valid_transformed)
xgb_accuracy = accuracy_score(y_valid, xgb_pred)

print(f"XGBoost validation accuracy:          {xgb_accuracy:.5f}")
print(f"Engineered CatBoost validation accuracy: {eng_accuracy:.5f}")

## 10. Ensemble experiment: CatBoost + XGBoost

In [ ]:
# Compare weighted probability averages on the same validation split.
# Class order is checked before combining probabilities.
assert np.array_equal(engineered_model.classes_, xgb_classifier.classes_)

ensemble_results = []
for cat_weight in [0.5, 0.6, 0.7, 0.8, 0.9]:
    ensemble_prob = cat_weight * eng_prob + (1 - cat_weight) * xgb_prob
    ensemble_pred = np.argmax(ensemble_prob, axis=1)
    ensemble_results.append({
        "catboost_weight": cat_weight,
        "xgboost_weight": 1 - cat_weight,
        "accuracy": accuracy_score(y_valid, ensemble_pred)
    })

display(pd.DataFrame(ensemble_results).sort_values(
    "accuracy", ascending=False
).reset_index(drop=True))
print(f"CatBoost alone: {eng_accuracy:.5f}")
print(f"XGBoost alone:  {xgb_accuracy:.5f}")
print("Based on the experiments so far, CatBoost alone remains the final model.")

## 11. Identifier investigation: `unit_id`

In [ ]:
# unit_id is a string (e.g. UNT-100018), so extract its numeric suffix before binning.
unit_id_num = train_df["unit_id"].str.extract(r"(\d+)$")[0].astype(int)
id_summary = train_df.assign(unit_id_num=unit_id_num).groupby("condition")["unit_id_num"].agg(
    ["count", "min", "max", "nunique", "mean"]
)
display(id_summary)

id_bins = pd.qcut(unit_id_num, q=10, duplicates="drop")
id_distribution = pd.crosstab(
    id_bins, train_df["condition"], normalize="index"
)
display(id_distribution.round(3))
print("Keep unit_id excluded unless a relationship is shown to generalize on validation data.")

## 12. Final submission — feature-engineered CatBoost only

In [ ]:
# Rebuild full-data features independently of the validation split.
X_full = train_df.drop(columns=["condition", "unit_id"]).copy()
y_full = train_df["condition"].copy()
X_full["unit_type"] = X_full["unit_type"].fillna("Unknown").astype(str)

X_test_final = test_df[X_full.columns].copy()
X_test_final["unit_type"] = X_test_final["unit_type"].fillna("Unknown").astype(str)

X_full_eng = add_features(X_full)
X_test_final_eng = add_features(X_test_final)

print("Full training feature shape:", X_full_eng.shape)
print("Test feature shape:", X_test_final_eng.shape)

In [ ]:
final_model = CatBoostClassifier(
    iterations=2000,
    depth=6,
    learning_rate=0.03,
    l2_leaf_reg=5,
    loss_function="MultiClass",
    eval_metric="Accuracy",
    cat_features=cat_features,
    random_seed=42,
    verbose=100,
    allow_writing_files=False
)

final_model.fit(X_full_eng, y_full)

In [ ]:
# Predict with CatBoost only.
test_predictions = final_model.predict(X_test_final_eng).ravel().astype(int)

prediction_df = pd.DataFrame({
    "unit_id": test_df["unit_id"],
    "condition": test_predictions
})

submission = sample_submission[["unit_id"]].merge(
    prediction_df,
    on="unit_id",
    how="left",
    validate="one_to_one"
)

assert len(submission) == len(sample_submission)
assert submission["condition"].notna().all()
assert submission["condition"].isin([0, 1, 2]).all()

submission["condition"] = submission["condition"].astype(int)
submission.to_csv("/content/submission.csv", index=False)

print("Submission created: /content/submission.csv")
print("Submission shape:", submission.shape)
print("Predicted class distribution:")
print(submission["condition"].value_counts().sort_index())
display(submission.head(10))

In [ ]:
from google.colab import files
files.download("/content/submission.csv")